# 05. Stage 3: Re-ranking (Maximal Marginal Relevance)

Notebook này xây dựng tầng đa dạng hóa danh sách đề xuất (Re-ranking) bằng giải thuật MMR để tối ưu hóa trải nghiệm người dùng, tránh sự trùng lặp thể loại quá mức.

---

### Phân tích Quyết định Thiết kế:
*   **Tại sao chọn Maximal Marginal Relevance (MMR)?**
    *   Các mô hình xếp hạng độ chính xác (như LightGBM) có xu hướng gợi ý một danh sách toàn các phim rất tương đồng nhau (ví dụ: 10 phim Hành động siêu anh hùng liên tiếp) vì chúng đều có điểm số cao. Điều này dễ gây nhàm chán. **MMR** cân bằng toán học giữa độ liên quan (score) và sự khác biệt (1 - similarity với các phim đã chọn trước đó trong danh sách). Lập trình viên dễ dàng điều chỉnh độ đa dạng qua siêu tham số lambda.
*   **Tại sao không chọn Deterministic Greedy Reranking?**
    *   Greedy thuần túy không có tham số để tinh chỉnh linh hoạt độ đa dạng và khó kết hợp trọng số điểm số gốc từ Ranker.
*   **Tại sao không chọn DPP (Determinant Point Processes)?**
    *   DPP là giải thuật tối ưu hóa xác suất rất mạnh nhưng độ phức tạp tính toán rất cao ($O(K^3)$), khó cài đặt hơn nhiều so với MMR ($O(K^2)$) vốn đơn giản, trực quan và chạy cực nhanh trên CPU.


In [1]:
import os
import pandas as pd
import numpy as np
import pickle
from sklearn.metrics.pairwise import cosine_similarity

# Load data và TF-IDF matrix để đo độ đa dạng thể loại
movies_df = pd.read_csv(os.path.join("..", "..", "data", "crawler", "movies_crawled.csv"))

with open("models/tfidf_matrix.pkl", "rb") as f:
    tfidf_matrix = pickle.load(f)


In [2]:
# 1. Định nghĩa giải thuật MMR
def maximal_marginal_relevance(item_scores, tfidf_matrix, lambda_param=0.7, top_k=10):
    if not item_scores:
        return []
        
    movie_id_to_idx = {row['movieId']: idx for idx, row in movies_df.iterrows()}
    
    candidates = [item[0] for item in item_scores]
    scores = np.array([item[1] for item in item_scores])
    
    if scores.max() != scores.min():
        scores_norm = (scores - scores.min()) / (scores.max() - scores.min())
    else:
        scores_norm = np.ones_like(scores)
        
    selected_items = []
    unselected_indices = list(range(len(candidates)))
    
    first_choice = np.argmax(scores_norm)
    selected_items.append(candidates[first_choice])
    unselected_indices.remove(first_choice)
    
    while len(selected_items) < top_k and unselected_indices:
        best_mmr = -1
        best_candidate_idx = -1
        
        selected_matrix_indices = [movie_id_to_idx[mid] for mid in selected_items]
        selected_vectors = tfidf_matrix[selected_matrix_indices]
        
        for idx in unselected_indices:
            candidate_id = candidates[idx]
            candidate_matrix_idx = movie_id_to_idx[candidate_id]
            candidate_vector = tfidf_matrix[candidate_matrix_idx]
            
            sim_with_selected = cosine_similarity(candidate_vector, selected_vectors).max()
            
            mmr_val = lambda_param * scores_norm[idx] - (1 - lambda_param) * sim_with_selected
            
            if mmr_val > best_mmr:
                best_mmr = mmr_val
                best_candidate_idx = idx
                
        selected_items.append(candidates[best_candidate_idx])
        unselected_indices.remove(best_candidate_idx)
        
    return selected_items


In [3]:
# 2. Thử nghiệm MMR đa dạng hóa
test_candidates = [
    (157336, 0.95),   # Interstellar
    (301528, 0.90),   # Toy Story 4
    (83533, 0.88),    # Avatar: Fire and Ash
    (1301310, 0.85),  # Zombies of the Third Reich
    (976912, 0.82),   # Graphic Desires
]

diversified = maximal_marginal_relevance(test_candidates, tfidf_matrix, lambda_param=0.5, top_k=3)
print("Gốc xếp hạng:", [movies_df[movies_df['movieId'] == mid]['title'].values[0] for mid, _ in test_candidates])
print("Sau MMR (Đa dạng):", [movies_df[movies_df['movieId'] == mid]['title'].values[0] for mid in diversified])


Gốc xếp hạng: ['Interstellar', 'Toy Story 4', 'Avatar: Fire and Ash', 'Zombies of the Third Reich', 'Graphic Desires']
Sau MMR (Đa dạng): ['Interstellar', 'Toy Story 4', 'Avatar: Fire and Ash']
